In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

json_parser = JsonOutputParser()
str_parser = StrOutputParser()

In [2]:
step_1_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an expert customer feedback analyst.

Extract the following information from the customer review:
1. complaint
2. product_or_feature
3. sentiment

Return the result as a JSON object.

{JSON_FORMATTING}"""
    ),
    (
        "user",
        "Customer review: {review}"
    )
]).partial(
    JSON_FORMATTING=json_parser.get_format_instructions()
)


In [4]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [5]:
step_1_chain = step_1_template | llm | json_parser

review = """
The phone battery drains in just two hours.
I am very disappointed with this product.
"""

step_1_output = step_1_chain.invoke({
    "review": review
})

print(step_1_output)

{'complaint': 'The phone battery drains in just two hours.', 'product_or_feature': 'phone battery', 'sentiment': 'negative'}


In [6]:
step_2_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an internal customer support ticket writer.

Create a concise support ticket for the internal team
using only the structured information provided.

Include:
- Ticket Title
- Complaint Summary
- Product or Feature
- Customer Sentiment
- Suggested Next Action

Do not use or assume any information beyond
the provided structured data."""
    ),
    (
        "user",
        """Complaint: {complaint}
Product or Feature: {product_or_feature}
Customer Sentiment: {sentiment}"""
    )
])

In [7]:
step_2_chain = step_2_template | llm | str_parser

step_2_output = step_2_chain.invoke(step_1_output)

print(step_2_output)

**Ticket Title:**  
Phone Battery Drains Rapidly (≈2 hrs)

**Complaint Summary:**  
Customer reports that the phone battery depletes in just two hours of use.

**Product or Feature:**  
Phone battery

**Customer Sentiment:**  
Negative

**Suggested Next Action:**  
Initiate battery performance diagnostics and consider a battery replacement or repair recommendation.


In [8]:
print("STEP 1: STRUCTURED COMPLAINT DATA")
print(step_1_output)

print("\nSTEP 2: SUPPORT TICKET SUMMARY")
print(step_2_output)

STEP 1: STRUCTURED COMPLAINT DATA
{'complaint': 'The phone battery drains in just two hours.', 'product_or_feature': 'phone battery', 'sentiment': 'negative'}

STEP 2: SUPPORT TICKET SUMMARY
**Ticket Title:**  
Phone Battery Drains Rapidly (≈2 hrs)

**Complaint Summary:**  
Customer reports that the phone battery depletes in just two hours of use.

**Product or Feature:**  
Phone battery

**Customer Sentiment:**  
Negative

**Suggested Next Action:**  
Initiate battery performance diagnostics and consider a battery replacement or repair recommendation.
